In [8]:
import logging
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as sf
from pyspark.sql.functions import month, days
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, StringType, TimestampNTZType

In [9]:
app_name = "Testing ETL Finance profiles"

spark = (
        SparkSession.builder
        .master("spark://spark-master:7077")
        .appName(app_name)
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
        .config("spark.sql.catalog.lakehouse_prod","org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.lakehouse_prod.type", "hive")
        .config("spark.sql.catalog.lakehouse_prod.uri","thrift://hive-metastore:9083")
        .config("spark.sql.catalog.lakehouse_prod.warehouse","s3a://lakehouse-prod-bucket/warehouse/")
        .config("spark.sql.catalog.lakehouse_prod.io-impl","org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .enableHiveSupport()
        .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("NOTE: SparkSession created successfully!")


NOTE: SparkSession created successfully!


In [3]:
spark

In [5]:
file_path = "./data/sp500_profiles.csv"
df = spark.read.csv(file_path, header=True, inferSchema=True)

In [9]:
df.toPandas()

,address1,city,state,zip,country,phone,website,industry,industryKey,industryDisp,...,regularMarketDayRange,marketState,regularMarketChangePercent,regularMarketPrice,displayName,trailingPegRatio,ipoExpectedDate,address2,fax,industrySymbol
0,2788 San Tomas Expressway,Santa Clara,CA,95051,United States,408-486-2000,https://www.nvidia.com,Semiconductors,semiconductors,Semiconductors,...,None,None,None,None,None,NaN,None,None,None,None
1,{'age': 58.0,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Ms. Colette M. Kress','title': 'Executive VP & CFO','totalPay': 1512641.0,'unexercisedValue': 0,'yearBorn': 1967.0},None,...,None,None,None,None,None,NaN,None,None,None,None
2,{'age': 70.0,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Ms. Debora Shoquist','title': 'Executive Vice President of Operati...,'totalPay': 1379071.0,'unexercisedValue': 0,'yearBorn': 1955.0},None,...,None,None,None,None,None,NaN,None,None,None,None
3,{'age': 58.0,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Mr. Timothy S. Teter J.D.','title': 'Executive VP,General Counsel & Secretary','totalPay': 1362989.0,'unexercisedValue': 0,'yearBorn': 1967.0},...,None,None,None,None,None,NaN,None,None,None,None
4,{'age': 70.0,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Mr. Ajay K. Puri','title': 'Executive Vice President of Worldwi...,'totalPay': 2313851.0,'unexercisedValue': 0,'yearBorn': 1955.0},None,...,None,None,None,None,None,NaN,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4799,{'age': 52.0,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Mr. Julian Delany','title': 'Executive VP & Chief Technology Off...,'totalPay': None,'unexercisedValue': 0,'yearBorn': 1973.0},None,...,None,None,None,None,None,NaN,None,None,None,None
4800,{'age': None,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Mr. Michael Florin','title': 'Senior VP & Head of Investor Relati...,'totalPay': None,'unexercisedValue': 0,'yearBorn': None},None,...,None,None,None,None,None,NaN,None,None,None,None
4801,{'age': None,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Mr. Arthur Bochner','title': 'Executive VP & Chief Communications...,'totalPay': None,'unexercisedValue': 0,'yearBorn': None},None,...,None,None,None,None,None,NaN,None,None,None,None
4802,{'age': None,'exercisedValue': 0,'fiscalYear': 2025.0,'maxAge': 1,'name': 'Ms. Anoushka Healy','title': 'Executive VP & Chief Strategy Officer','totalPay': None,'unexercisedValue': 0,'yearBorn': None},None,...,None,None,None,None,None,NaN,None,None,None,None


In [10]:
df.printSchema()

root
 |-- address1: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- country: string (nullable = true)
 |-- phone: string (nullable = true)
 |-- website: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- industryKey: string (nullable = true)
 |-- industryDisp: string (nullable = true)
 |-- sector: string (nullable = true)
 |-- sectorKey: string (nullable = true)
 |-- sectorDisp: string (nullable = true)
 |-- longBusinessSummary: string (nullable = true)
 |-- fullTimeEmployees: string (nullable = true)
 |-- companyOfficers: string (nullable = true)
 |-- auditRisk: string (nullable = true)
 |-- boardRisk: string (nullable = true)
 |-- compensationRisk: string (nullable = true)
 |-- shareHolderRightsRisk: string (nullable = true)
 |-- overallRisk: string (nullable = true)
 |-- governanceEpochDate: string (nullable = true)
 |-- compensationAsOfEpochDate: string (nullable = true)
 |-

In [11]:
df.groupBy(df.symbol).count().sort("symbol").show()

[Stage 7:>                                                          (0 + 1) / 1]

+--------------------+-----+
|              symbol|count|
+--------------------+-----+
|                NULL| 4301|
|                 CNP|    1|
|                 ETN|    1|
|                FISV|    1|
|               False|  430|
|                HIGH|   56|
|                MPWR|    1|
|Nasdaq Real Time ...|    4|
|                PYPL|    1|
|                 TMO|    1|
|                True|    7|
+--------------------+-----+



In [3]:
# Define the S3 path (using s3a protocol)
s3_bucket_path = "s3a://datalake-landing/profiles/sp500_profiles_20260308.parquet"

# Read CSV data from S3 into a DataFrame
df = spark.read.parquet(s3_bucket_path, header=True, inferSchema=True)

# Show data
df.toPandas()

26/03/10 17:19:40 ERROR TransportClient: Failed to send RPC RPC 8557441231021552839 to /172.22.0.10:51834: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
26/03/10 17:19:40 ERROR TransportClient: Failed to send RPC RPC 8741911515642610483 to /172.22.0.9:54960: io.netty.channel.StacklessClosedChannelException
io.netty.channel.StacklessClosedChannelException
	at io.netty.channel.AbstractChannel$AbstractUnsafe.write(Object, ChannelPromise)(Unknown Source)
                                                                                

,address1,city,state,zip,country,phone,website,industry,industryKey,industryDisp,...,isEarningsDateEstimate,epsTrailingTwelveMonths,epsForward,epsCurrentYear,displayName,trailingPegRatio,ipoExpectedDate,address2,fax,industrySymbol
0,2788 San Tomas Expressway,Santa Clara,CA,95051,United States,408-486-2000,https://www.nvidia.com,Semiconductors,semiconductors,Semiconductors,...,False,4.91,10.74220,8.24943,NVIDIA,1.0750,None,None,None,None
1,One Apple Park Way,Cupertino,CA,95014,United States,(408) 996-1010,https://www.apple.com,Consumer Electronics,consumer-electronics,Consumer Electronics,...,True,7.90,9.28890,8.49805,Apple,2.3044,None,None,None,None
2,One Microsoft Way,Redmond,WA,98052-6399,United States,425 882 8080,https://www.microsoft.com,Software - Infrastructure,software-infrastructure,Software - Infrastructure,...,True,15.99,18.84224,16.72920,Microsoft,1.3341,None,None,None,None
3,410 Terry Avenue North,Seattle,WA,98109-5210,United States,206 266 1000,https://www.amazon.com,Internet Retail,internet-retail,Internet Retail,...,True,7.16,9.33864,7.72036,Amazon.com,1.9491,None,None,None,None
4,1600 Amphitheatre Parkway,Mountain View,CA,94043,United States,650-253-0000,https://abc.xyz,Internet Content & Information,internet-content-information,Internet Content & Information,...,True,10.81,13.42320,11.57674,None,2.2467,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,200 Oceangate,Long Beach,CA,90802,United States,(562) 435-3666,https://www.molinahealthcare.com,Healthcare Plans,healthcare-plans,Healthcare Plans,...,False,8.91,10.23591,6.95756,Molina Healthcare,0.5679,None,Suite 100,None,None
499,7501 West Memorial Road,Oklahoma City,OK,73142,United States,405 722 6900,https://www.paycom.com,Software - Application,software-application,Software - Application,...,True,8.07,11.40768,10.31768,Paycom Software,1.2254,None,None,None,None
500,8750 North Central Expressway,Dallas,TX,75231,United States,214 576 9352,https://mtch.com,Internet Content & Information,internet-content-information,Internet Content & Information,...,True,2.38,4.03037,3.62886,Match,0.3194,None,Suite 1400,None,None
501,599 South Rivershore Lane,Eagle,ID,83616,United States,208 938 1047,https://www.lambweston.com,Packaged Foods,packaged-foods,Packaged Foods,...,False,2.76,3.01460,2.74724,Lamb Weston,0.6477,None,None,None,None


In [4]:
duplicate_rows = df.count() - df.dropDuplicates().count()
duplicate_rows

0

In [5]:
df.groupBy("industry").count().sort("count", ascending=False).show()

+--------------------+-----+
|            industry|count|
+--------------------+-----+
|Utilities - Regul...|   23|
|Specialty Industr...|   17|
|Software - Applic...|   16|
|Software - Infras...|   15|
|      Semiconductors|   13|
|    Asset Management|   13|
| Aerospace & Defense|   12|
|Diagnostics & Res...|   11|
|     Medical Devices|   10|
|       Entertainment|   10|
|Information Techn...|   10|
|       Oil & Gas E&P|   10|
| Specialty Chemicals|    9|
|Drug Manufacturer...|    9|
|Medical Instrumen...|    9|
|Financial Data & ...|    9|
|    Banks - Regional|    9|
|Insurance - Prope...|    8|
|      Packaged Foods|    8|
|   Computer Hardware|    7|
+--------------------+-----+
only showing top 20 rows



In [9]:
df.select('*').show(5)

+--------------------+-------------+-----+----------+-------------+--------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+---------+---------+----------------+---------------------+-----------+-------------------+-------------------------+--------------------+-------------+------+---------+-------------+------+--------+--------+--------------------------+-----------------+-------------------+--------------------+------------+-------------+--------------+-----------+------------------------+-----+----------+---------+---------+-------------------+-------------+-------------------+-----------------------+------+------+-------+-------+-------------+-------------------+---------------+----------------+-----------+----------+----------------------------+---------------+--------------------+--------------------------+----------

In [7]:
import great_expectations as gx
context = gx.get_context()

In [8]:
data_source_name = "test_dataset"
data_source = context.data_sources.add_spark(name=data_source_name)


In [11]:
validator = data_source.read_dataframe(df)
validator.expect_column_values_to_not_be_null("symbol")

AttributeError: 'SparkDatasource' object has no attribute 'read_dataframe'

In [4]:
spark.sql('''SHOW DATABASES;''').show()

+---------+
|namespace|
+---------+
|bronze_db|
|  default|
|  gold_db|
|silver_db|
+---------+



In [11]:
spark.sql('''SHOW TABLES;''').show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



In [12]:
spark.sql(''' DROP TABLE lakehouse_prod.bronze_db.bronze_country_iso_profiles;''').show()

++
||
++
++



26/03/23 16:12:51 ERROR StandaloneSchedulerBackend: Application has been killed. Reason: Master removed our application: KILLED
26/03/23 16:12:51 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exiting due to error from cluster scheduler: Master removed our application: KILLED
	at org.apache.spark.errors.SparkCoreErrors$.clusterSchedulerError(SparkCoreErrors.scala:291)
	at org.apache.spark.scheduler.TaskSchedulerImpl.error(TaskSchedulerImpl.scala:981)
	at org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend.dead(StandaloneSchedulerBackend.scala:165)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint.markDead(StandaloneAppClient.scala:263)
	at org.apache.spark.deploy.client.StandaloneAppClient$ClientEndpoint$$anonfun$receive$1.applyOrElse(StandaloneAppClient.scala:170)
	at org.apache.spark.rpc.netty.Inbox.$anonfun$process$1(Inbox.scala:115)
	at org.apache.spark.rpc.netty.Inbox.safelyCall(Inbox.scala:213)
	at org.apache.spark.rpc.netty.Inbox.proce

In [7]:
spark.stop()